In [54]:
from langchain_openai.chat_models import ChatOpenAI
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
import copy
import sys
sys.path.append("..")
import config
import numpy as np

#### PYPDF Loader

In [2]:
loaderpdf = PyPDFLoader("dataScience.pdf")

In [8]:
pagesPDF = loaderpdf.load()

In [11]:
pagesPDFCut = copy.deepcopy(pagesPDF)

In [15]:
for i in pagesPDFCut:
    i.page_content = " ".join(i.page_content.split()) # remove \n

#### DOCX2TXT Loader

In [17]:
from langchain_community.document_loaders import Docx2txtLoader

In [18]:
loaderdocx = Docx2txtLoader("dataScience.docx")

In [20]:
pages = loaderdocx.load()
for i in range(len(pages)):
    pages[i].page_content = " ".join(pages[i].page_content.split())

#### Document splitting

In [24]:
from langchain_text_splitters.character import CharacterTextSplitter

In [21]:
len(pages[0].page_content)

8259

In [26]:
charSplitter = CharacterTextSplitter(separator = "",
                                     chunk_size = 500, #500 chars per chunk
                                     chunk_overlap = 0)

In [28]:
pagesCharSplit = charSplitter.split_documents(pages)
len(pagesCharSplit)

17

In [30]:
8259 / 500

16.518

In [32]:
0.518 * 500 == len(pagesCharSplit[-1].page_content)

True

In [39]:
charSplitter = CharacterTextSplitter(separator = "", #default \n\n
                                     chunk_size = 500,
                                     chunk_overlap = 50)

pagesCharSplit = charSplitter.split_documents(pages)

In [40]:
pagesCharSplit[0].page_content

'Analysis vs Analytics Alright! So… Let’s discuss the not-so-obvious differences between the terms analysis and analytics. Due to the similarity of the words, some people believe they share the same meaning, and thus use them interchangeably. Technically, this isn’t correct. There is, in fact, a distinct difference between the two. And the reason for one often being used instead of the other is the lack of a transparent understanding of both. So, let’s clear this up, shall we? First, we will star'

In [41]:
pagesCharSplit[1].page_content

'let’s clear this up, shall we? First, we will start with analysis. Consider the following… You have a huge dataset containing data of various types. Instead of tackling the entire dataset and running the risk of becoming overwhelmed, you separate it into easier to digest chunks and study them individually and examine how they relate to other parts. And that’s analysis in a nutshell. One important thing to remember, however, is that you perform analyses on things that have already happened in the'

#### Text embedding

In [43]:
from langchain_openai.embeddings import OpenAIEmbeddings

In [45]:
embedding = OpenAIEmbeddings(
    model = "openai/text-embedding-ada-002",
    api_key=config.apiKey2,
    base_url="https://openrouter.ai/api/v1"
)

In [48]:
vector1 = embedding.embed_query(pagesCharSplit[3].page_content)

In [49]:
vector2 = embedding.embed_query(pagesCharSplit[5].page_content)

In [50]:
vector3 = embedding.embed_query(pagesCharSplit[18].page_content)

In [51]:
vector1

[-0.00431244820356369,
 0.004348385613411665,
 0.004122961778193712,
 -0.026501955464482307,
 -0.024933792650699615,
 0.012466896325349808,
 -0.02856670320034027,
 -0.0199679434299469,
 0.006086432840675116,
 -0.02260768413543701,
 0.01496288925409317,
 0.017197521403431892,
 -0.012414623983204365,
 0.005423230584710836,
 -0.017092976719141006,
 -0.023548580706119537,
 0.0358847975730896,
 -0.00397267984226346,
 0.021392356604337692,
 -0.017929330468177795,
 -0.03779272735118866,
 0.028723519295454025,
 -0.003463026601821184,
 -0.030553042888641357,
 -0.014858344569802284,
 0.0030579178128391504,
 0.0022901713382452726,
 -0.023025861009955406,
 0.003926941659301519,
 -0.02063441090285778,
 0.003443424589931965,
 0.008637964725494385,
 -0.015054364688694477,
 0.0015395766822621226,
 -0.010460954159498215,
 0.0043320502154529095,
 0.008938529528677464,
 0.009729145094752312,
 0.014727664180099964,
 0.012192467227578163,
 0.011408385820686817,
 -0.0007526365807279944,
 -0.0256263967603445

In [53]:
len(vector1), len(vector2), len(vector3) # dimensions

(1536, 1536, 1536)

In [55]:
np.dot(vector1, vector2)

np.float64(0.8769080437509426)

In [56]:
np.dot(vector1, vector3), np.dot(vector2, vector3)

(np.float64(0.773089521633717), np.float64(0.7744441810646001))

In [59]:
# magnitude of 1
np.linalg.norm(vector1), np.linalg.norm(vector2), np.linalg.norm(vector3)

(np.float64(0.9999998929504711),
 np.float64(0.9999999154650702),
 np.float64(0.9999999739575571))

#### Chroma VectorStore

In [68]:
from langchain_chroma import Chroma
from langchain_core.documents import Document

In [66]:
vectorstore = Chroma.from_documents(documents = pagesCharSplit,
                                    embedding = embedding,
                                   persist_directory = "./chromavectorstore") #allows to retrieve embeddings when kernel is restarted

In [67]:
vectorstoreFromDir = Chroma(persist_directory = "./chromavectorstore",
                           embedding_function = embedding)

In [69]:
vectorstoreFromDir.get()

{'ids': ['b796df06-616e-4092-b784-8d4e9a67d42a',
  '4a9480b9-876d-4cce-89d3-a72dd15bb065',
  '58b61c08-c376-4a0b-aa25-8b6bad8b93c3',
  '3d252eec-303e-4f17-9ec9-bb6b71651803',
  '5fcd14ac-690e-4d2e-8d3d-7156083d6847',
  '1e0263f6-44fd-4d4b-9d4c-de2e62d5f1de',
  '48fd1872-b78a-4fea-8e7a-ef5bdd7447c0',
  '93f3c75b-7771-4972-8586-d93d825729e8',
  'e7f72728-b41d-4422-bca7-3a3877328cce',
  '57bcca04-c9f2-40c4-b737-2718cd63cbae',
  '1792ea02-48a9-41ae-af20-503abbe4e330',
  '7201066b-d87d-4972-a3b2-c75b08214786',
  '1c703098-7b73-4f99-a22d-f85f282e30c4',
  'f922e173-03af-4ee8-a980-15a785aab5e5',
  '60969122-b296-4c72-afb6-d267f1831950',
  '4d03f540-0c89-4a91-bdc2-aebd29cb8253',
  '4753e63c-6fb0-4702-a58a-55b135520cae',
  '9050ac07-a299-4120-a287-dc98eb0995a9',
  '655bc414-3dbd-4241-ac2a-d406f99a3de9',
  '68df4c25-b40e-412d-a9b4-61ff0c70fa7d',
  'f714a781-c0b9-4c4f-9fe0-f0afc2afa59b',
  '7d827bee-bda5-4689-9c6b-b5bf1a5d59c6',
  '6f0389e5-38c9-4ffb-91fc-045714ae0816',
  '140f0bdf-8eac-4055-a8da-

In [70]:
# vectorstoreFromDir.add_documents([addedDoc])
# vectorstoreFromDir.update_document(document_id = "", document = updatedDoc)
# vectorstoreFromDir.delete("")

#### Similarity search

In [83]:
q = "What programming languages do data scientists use?"

In [84]:
retrievedDocs = vectorstoreFromDir.similarity_search(query = q,
                                                     k = 5 # num of retrieved docs
                                                    )

In [85]:
for i in retrievedDocs:
    print(f"Page Content: {i.page_content}\n-------\n")

Page Content: open-source languages like R and Python. Either way, R, Python, and MATLAB, combined with SQL, cover most of the tools used when working with traditional data, BI, and conventional data science. What about big data? Apart from R and Python, people working in this area are often proficient in other languages like Java or Scala. These two have not been developed specifically for doing statistical analyses, however they turn out to be very useful when combining data from multiple sources. All right
-------

Page Content: open-source languages like R and Python. Either way, R, Python, and MATLAB, combined with SQL, cover most of the tools used when working with traditional data, BI, and conventional data science. What about big data? Apart from R and Python, people working in this area are often proficient in other languages like Java or Scala. These two have not been developed specifically for doing statistical analyses, however they turn out to be very useful when combining

#### Maximal Marginal Relevance (MMR) search algorithm 
- similarity + diversity
- λ e [0, 1]
- λ closer to 0 => emphasis on diversity

In [91]:
retrievedDocs = vectorstoreFromDir.max_marginal_relevance_search(
    query = q,
    k = 3,
    lambda_mult = 0.1,
   #filter = {"Lecture Title": "Programming Languages & Software Employed in Data Science"},
)

In [90]:
for i in retrievedDocs:
    print(f"Page Content: {i.page_content}\n-------\n")

Page Content: open-source languages like R and Python. Either way, R, Python, and MATLAB, combined with SQL, cover most of the tools used when working with traditional data, BI, and conventional data science. What about big data? Apart from R and Python, people working in this area are often proficient in other languages like Java or Scala. These two have not been developed specifically for doing statistical analyses, however they turn out to be very useful when combining data from multiple sources. All right
-------

Page Content: the next video which aims to simplify these, as well as many more with a fantastic diagram. So, let’s move on! Programming Languages & Software Employed in Data Science - All the Tools You Need Alright! So… How are the techniques used in data, business intelligence, or predictive analytics applied in real life? Certainly, with the help of computers. You can basically split the relevant tools into two categories—programming languages and software. Knowing a p

#### vectorstore backed retriever

In [95]:
retriever = vectorstoreFromDir.as_retriever(search_type = "mmr",
                                           search_kwargs = {"k":3, "lambda_mult": 0.7},)

In [96]:
retriever

VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x000001F4DBE01F30>, search_type='mmr', search_kwargs={'k': 3, 'lambda_mult': 0.7})

In [97]:
q = "What sofware do data scientists use?"
retrievedDocs = retriever.invoke(q)

In [99]:
retrievedDocs

[Document(id='7201066b-d87d-4972-a3b2-c75b08214786', metadata={'source': 'dataScience.docx'}, page_content='open-source languages like R and Python. Either way, R, Python, and MATLAB, combined with SQL, cover most of the tools used when working with traditional data, BI, and conventional data science. What about big data? Apart from R and Python, people working in this area are often proficient in other languages like Java or Scala. These two have not been developed specifically for doing statistical analyses, however they turn out to be very useful when combining data from multiple sources. All right'),
 Document(id='655bc414-3dbd-4241-ac2a-d406f99a3de9', metadata={'source': 'dataScience.docx'}, page_content='u a good idea about the level of applicability of the most frequently used programming and software tools in the field of data science. Thank you for watching!'),
 Document(id='93f3c75b-7771-4972-8586-d93d825729e8', metadata={'source': 'dataScience.docx'}, page_content='the next 

#### Generation

In [100]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser as strp

In [101]:
temp = """
Answer the following question:
{q}
To answer the question, use only the following context:
{context}
"""

In [102]:
promptTemplate = PromptTemplate.from_template(temp)

chat = ChatOpenAI(
    model = "openai/gpt-4",
    max_tokens = 250,
    seed = 365,
    api_key=config.apiKey2,
    base_url="https://openrouter.ai/api/v1"
)

##### stuffing - all retrieved docs in a prompt

In [112]:
chain = {"context": retriever,
        "q": RunnablePassthrough()} | promptTemplate | chat | strp()

In [114]:
print(chain.invoke(q))

Data scientists use open-source languages like R and Python, as well as MATLAB combined with SQL for traditional data, BI, and conventional data science. For big data, they may also be proficient in languages like Java or Scala.


##### Document Refinement => alternative
-  fix context window limit
-  passes docs one at a time then updates the response with additional info
-  more calls to the llm